# Der KI Neues beibringen

WAS IST WAS Wissenschaften easy: Künstliche Intelligenz

**Tipp:** Du kannst alles ändern und ausprobieren. Wenn du deine Änderungen behalten willst, wähle oben *Datei → Kopie in Drive speichern*.

Die vortrainierte KI kennt keine Handzeichen. Hier bringst du ihr Schere, Stein, Papier bei: Ihre gelernten Schichten bleiben eingefroren, nur eine neue Ausgabe mit drei Knoten lernt dazu. Genau so arbeitet auch Teachable Machine.

Klicke links neben der grauen Code-Zelle auf das Dreieck ▶ oder drücke Strg + Enter (am Mac Cmd + Enter).

Als Erstes installierst du ein kleines Hilfspaket. Colab braucht es, um die Fotos der Handzeichen zu laden.

In [ ]:
!pip install -q importlib-resources

Jetzt trainierst du die KI. Der Seed in Zeile 3 sorgt dafür, dass derselbe Zufall herauskommt. Deine Zahlen können trotzdem in der letzten Stelle etwas von denen im Buch abweichen.

In [ ]:
import keras, tensorflow_datasets as tfds
from keras.applications import mobilenet_v2 as mn
keras.utils.set_random_seed(42)
ueben, test = tfds.load("rock_paper_scissors",
    split=["train", "test"], as_supervised=True)
def anpassen(bild, nummer):
    bild = keras.ops.cast(bild, "float32")
    bild = keras.ops.image.resize(bild, (160, 160))
    return mn.preprocess_input(bild), nummer
ueben = ueben.map(anpassen).shuffle(1000).batch(32)
test = test.map(anpassen).batch(32)
basis = mn.MobileNetV2(input_shape=(160, 160, 3),
    include_top=False, weights="imagenet")
basis.trainable = False
ki = keras.Sequential([basis,
    keras.layers.GlobalAveragePooling2D(),
    keras.layers.Dense(3, activation="softmax")])
ki.compile(optimizer="adam", metrics=["accuracy"],
           loss="sparse_categorical_crossentropy")
ki.fit(ueben, epochs=3)
ki.evaluate(test)

## Deine eigene Hand

Führe zuerst die Zelle oben aus, damit die KI trainiert ist. Mach dann ein Foto von deiner Hand, die Schere, Stein oder Papier zeigt, und wähle es hier aus. Die KI hat nur Hände gesehen, die am Computer erzeugt wurden, alle vor weißem Hintergrund. Erkennt sie trotzdem deine echte Hand? Am besten klappt es vor einer hellen Wand oder einem Blatt Papier.

Fotografiere nur deine Hand, kein Gesicht.

In [ ]:
import numpy as np
from google.colab import files
zeichen = ["Stein", "Papier", "Schere"]
name = list(files.upload())[0]
foto = keras.utils.load_img(name, target_size=(160, 160),
                            interpolation="bilinear")
x = keras.utils.img_to_array(foto)
werte = ki.predict(mn.preprocess_input(np.array([x])))[0]
for wort, p in zip(zeichen, werte):
    print(wort, round(p * 100, 1), "%")

**Probier’s aus:** Ändere in Zeile 13 `weights="imagenet"` in `weights=None` und führe die Zelle noch einmal aus. Wie viel ist das Vorwissen wert?